In [ ]:
# ─── 1. Install ───────────────────────────────────────────────────────────────
!pip install -q datasets scikit-learn

# ─── 2. Mount Drive ───────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/TFIDF_SVM_IFND"
import os, time, json
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── 3. Imports ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import joblib
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

# ─── 4. Load dataset ──────────────────────────────────────────────────────────
print("⏳ Loading dataset IFND-multimodal...")
hf_dataset = load_dataset("Nhat243/IFND-multimodal")
print(hf_dataset)

# Kiểm tra label distribution
train_labels_list = hf_dataset['train']['label']
print(f"📊 Label distribution - Train: REAL={list(train_labels_list).count(1)}, FAKE={list(train_labels_list).count(0)}")

# ─── 5. Extract text & labels (chuyển sang numpy array) ───────────────────────
print("⏳ Extracting text...")
train_texts = hf_dataset['train']['text']
train_labels = np.array(hf_dataset['train']['label'])  # 🔥 Chuyển sang numpy array
val_texts = hf_dataset['validation']['text']
val_labels = np.array(hf_dataset['validation']['label'])  # 🔥 Chuyển sang numpy array
test_texts = hf_dataset['test']['text']
test_labels = np.array(hf_dataset['test']['label'])  # 🔥 Chuyển sang numpy array

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")
print(f"Train labels type: {type(train_labels[0])}")  # Kiểm tra

# ─── 6. TF-IDF Vectorization ──────────────────────────────────────────────────
print("⏳ Fitting TF-IDF...")
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True
)
X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)
X_test = vectorizer.transform(test_texts)
print(f"Feature matrix: {X_train.shape}")

# ─── 7. Train SVM (KHÔNG dùng CalibratedClassifierCV để tránh lỗi) ─────────────
print("⏳ Training SVM...")
t0 = time.perf_counter()
svm = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm.fit(X_train, train_labels)
train_time = time.perf_counter() - t0
print(f"✅ Done in {train_time:.1f}s")

# ─── 8. Validation (dùng decision_function cho AUC) ───────────────────────────
val_preds = svm.predict(X_val)
# LinearSVC không có predict_proba, dùng decision_function cho AUC
val_scores = svm.decision_function(X_val)

val_acc = accuracy_score(val_labels, val_preds)
val_precision, val_recall, val_f1, _ = precision_recall_fscore_support(
    val_labels, val_preds, average="binary")

# Tính AUC với decision_function
from sklearn.metrics import roc_curve, auc
val_auc = roc_auc_score(val_labels, val_scores)

print(f"\n📊 Validation Results:")
print(f"   Accuracy : {val_acc*100:.2f}%")
print(f"   Precision: {val_precision*100:.2f}%")
print(f"   Recall   : {val_recall*100:.2f}%")
print(f"   F1       : {val_f1*100:.2f}%")
print(f"   AUC      : {val_auc:.4f}")

# ─── 9. Test Evaluation ───────────────────────────────────────────────────────
print("\n📊 Evaluating on test set...")
test_preds = svm.predict(X_test)
test_scores = svm.decision_function(X_test)

test_acc = accuracy_score(test_labels, test_preds)
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average="binary")
test_auc = roc_auc_score(test_labels, test_scores)

print(f"\n{'='*50}")
print(f"📊 TEST RESULTS - TF-IDF + SVM on IFND")
print(f"{'='*50}")
print(f"   Accuracy : {test_acc*100:.2f}%")
print(f"   Precision: {test_precision*100:.2f}%")
print(f"   Recall   : {test_recall*100:.2f}%")
print(f"   F1 Score : {test_f1*100:.2f}%")
print(f"   AUC      : {test_auc:.4f}")
print(f"{'='*50}")

# ─── 10. Latency Measurement ──────────────────────────────────────────────────
print("\n⏱️ Measuring inference latency...")
dummy_texts = ["This is a sample news headline for inference measurement on IFND dataset."]

# Warmup
for _ in range(50):
    x = vectorizer.transform(dummy_texts)
    svm.predict(x)

latencies = []
for _ in range(200):
    t0 = time.perf_counter()
    x = vectorizer.transform(dummy_texts)
    svm.predict(x)
    latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
latency_mean = np.mean(latencies)
latency_p50 = np.percentile(latencies, 50)
latency_p95 = np.percentile(latencies, 95)

print(f"   Mean: {latency_mean:.2f} ms")
print(f"   P50 : {latency_p50:.2f} ms")
print(f"   P95 : {latency_p95:.2f} ms")

# ─── 11. Model Info ───────────────────────────────────────────────────────────
model_path = os.path.join(SAVE_DIR, "tfidf_svm.joblib")
joblib.dump({"vectorizer": vectorizer, "clf": svm}, model_path)
model_size_mb = os.path.getsize(model_path) / 1024**2

# Số lượng features
n_features = X_train.shape[1]
n_params = n_features * 2  # 2 classes (weights + bias gần đúng)

print(f"\n💾 Model Info:")
print(f"   Feature dimension: {n_features:,}")
print(f"   Equivalent params: {n_params:,}")
print(f"   Model size: {model_size_mb:.1f} MB")
print(f"   Platform: CPU-only (no VRAM usage)")

# ─── 12. Save Results JSON ────────────────────────────────────────────────────
results = {
    "Dataset": "IFND-multimodal",
    "Model": "TF-IDF + Linear SVM",
    "Method": "Traditional ML Baseline",
    "Label_Mapping": "1=REAL, 0=FAKE",
    "Preprocessing": {
        "Vectorizer": "TF-IDF",
        "Max_Features": 10000,
        "Ngram_Range": "(1,2)",
        "Sublinear_TF": True
    },
    "Validation_Results": {
        "Accuracy (%)": round(val_acc * 100, 2),
        "Precision (%)": round(val_precision * 100, 2),
        "Recall (%)": round(val_recall * 100, 2),
        "F1 (%)": round(val_f1 * 100, 2),
        "AUC": round(val_auc, 4)
    },
    "Test_Results": {
        "Accuracy (%)": round(test_acc * 100, 2),
        "Precision (%)": round(test_precision * 100, 2),
        "Recall (%)": round(test_recall * 100, 2),
        "F1 (%)": round(test_f1 * 100, 2),
        "AUC": round(test_auc, 4)
    },
    "Latency_ms": {
        "Mean": round(latency_mean, 2),
        "P50": round(latency_p50, 2),
        "P95": round(latency_p95, 2)
    },
    "Hardware_Stats": {
        "Platform": "CPU",
        "VRAM_GB": 0,
        "Model_Size_MB": round(model_size_mb, 1),
        "Feature_Dimension": n_features,
        "Equivalent_Params": n_params,
        "Training_Time_Seconds": round(train_time, 2)
    }
}

# Lưu JSON
json_path = os.path.join(SAVE_DIR, "results_IFND_TFIDF_SVM.json")
with open(json_path, "w") as f:
    json.dump(results, f, indent=4)

# Lưu CSV
df_results = pd.DataFrame([results["Test_Results"]])
df_results.to_csv(os.path.join(SAVE_DIR, "results_IFND_TFIDF_SVM.csv"))

# ─── 13. Display Summary ──────────────────────────────────────────────────────
print("\n" + "="*60)
print("📊 KẾT QUẢ TỔNG HỢP TF-IDF + SVM trên IFND")
print("="*60)
print(f"📍 Test Set: IFND-multimodal")
print(f"   Accuracy : {results['Test_Results']['Accuracy (%)']}%")
print(f"   Precision: {results['Test_Results']['Precision (%)']}%")
print(f"   Recall   : {results['Test_Results']['Recall (%)']}%")
print(f"   F1 Score : {results['Test_Results']['F1 (%)']}%")
print(f"   AUC      : {results['Test_Results']['AUC']}")
print(f"\n⚡ Performance:")
print(f"   Latency (P50): {latency_p50:.2f} ms/sample")
print(f"   Model Size   : {model_size_mb:.1f} MB")
print(f"   Platform     : CPU-only")
print("="*60)

print(f"\n✅ Results saved to: {SAVE_DIR}")
print(f"   - results_IFND_TFIDF_SVM.json")
print(f"   - results_IFND_TFIDF_SVM.csv")
print(f"   - tfidf_svm.joblib")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ Loading dataset IFND-multimodal...
DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 8416
    })
    validation: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 1052
    })
    test: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 1053
    })
})
📊 Label distribution - Train: REAL=6447, FAKE=1969
⏳ Extracting text...
Train: 8416 | Val: 1052 | Test: 1053
Train labels type: <class 'numpy.int64'>
⏳ Fitting TF-IDF...
Feature matrix: (8416, 10000)
⏳ Training SVM...
✅ Done in 0.0s

📊 Validation Results:
   Accuracy : 98.19%
   Precision: 97.93%
   Recall   : 99.75%
   F1       : 98.83%
   AUC      : 0.9932

📊 Evaluating on test set...

📊 TEST RESULTS - TF-IDF + SVM on IFND
   Accuracy : 98.10%
   Precision: 97.93%
   Recall   : 99.63%
   F1 Score 

In [ ]:
print("⏳ Đang ngắt kết nối phiên làm việc để giải phóng GPU...")
from google.colab import runtime
time.sleep(10) # Đợi đồng bộ Drive
runtime.unassign()

⏳ Đang ngắt kết nối phiên làm việc để giải phóng GPU...
